# Run mini-infer Pallas TPU kernels on a free Colab TPU

Needs only a Google account (no identity verification, no card).

1. **Runtime -> Change runtime type -> TPU**.
2. **Runtime -> Run all**.

The cell below clones the public `tpu-pallas-backend` branch and runs every
kernel (dense, paged decode, paged prefill; MHA + GQA) with `interpret=False`,
checking each against a NumPy reference. A real TPU run prints
`devices: [TpuDevice(...)]` and ends with `ALL PASS`. It refuses to fall back
to CPU, so a green run genuinely ran on the TPU.

In [ ]:
import os
import subprocess
import sys

import jax

print("jax", jax.__version__, "devices:", jax.devices())
if not any(getattr(d, "platform", "") == "tpu" for d in jax.devices()):
    raise SystemExit(
        "No TPU. Runtime -> Change runtime type -> TPU, then Runtime -> Run all."
    )

DEST = "/content/mini-infer"
if not os.path.isdir(DEST):
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "tpu-pallas-backend",
         "https://github.com/JonathanBerhe/mini-infer.git", DEST],
        check=True,
    )
sys.path.insert(0, os.path.join(DEST, "src"))
sys.path.insert(0, os.path.join(DEST, "scripts"))
os.chdir(DEST)

import run_tpu_pallas_kernels as runner

print("exit code:", runner.main())
